In [1]:
import os

# Caminho absoluto da pasta onde o script está sendo executado
diretorio_atual = os.getcwd()

print("Diretório atual:", diretorio_atual)

Diretório atual: /workspaces/IC-RNA-2025/Peaks-dataset-article/media_ponderada


In [ ]:
import os
import pandas as pd
import numpy as np
from itertools import combinations, islice, permutations
from sklearn.metrics import r2_score
from random import shuffle
import random 

# Diretórios dos modelos
diretorio_25_model = "/workspaces/IC-RNA-2025/Peaks-dataset-article/redes-ensemble-s/metricas_ensemble_50_redes_1/25_model"
diretorio_1000_model = "/workspaces/IC-RNA-2025/Peaks-dataset-article/redes-ensemble-s/metricas_ensemble_50_redes_1/1000_model"

# Listagem dos arquivos
file_list_25_model = [os.path.join(diretorio_25_model, f) for f in os.listdir(diretorio_25_model) if f.endswith(".xlsx")]
file_list_1000_model = [os.path.join(diretorio_1000_model, f) for f in os.listdir(diretorio_1000_model) if f.endswith(".xlsx")]

lambda_reg = 5e-6
lr = 3e-5
n_epocas = 5000000
patience = 500
min_delta = 1e-2

def carrega_Z_e_Zpreds(conjunto_25, arquivo_Z):
    df_list = [pd.read_excel(f) for f in conjunto_25]
    z_preds = np.hstack([df['Z_pred'].values.reshape(-1,1) for df in df_list])
    Z = pd.read_excel(arquivo_Z)['Z'].values.reshape(-1,1)
    return Z, z_preds, Z.shape[0]


def otimiza_pesos(Z, z_preds, N):
    a = np.zeros(z_preds.shape[1])
    best_mse_25 = -np.inf
    best_w = None
    patience_counter = 0

    for _ in range(n_epocas):
        w = np.exp(a) / np.sum(np.exp(a))
        yhat = z_preds @ w

        residuo = Z.flatten() - yhat
        mse = np.mean(residuo**2)

        loss = mse + lambda_reg * np.sum(a**2)

        gmse = (-2.0 / N) * z_preds.T @ residuo
        s = gmse @ w
        grad_a = w * (gmse - s) + 2.0 * lambda_reg * a

        a -= lr * grad_a

        # critério: MAIOR mse_25
        if mse > best_mse_25 + min_delta:
            best_mse_25 = mse
            best_w = w.copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    return best_w


# Função para cálculo de MSE usando 1000_model
def calcula_mse_sup(conjunto, best_w):
    df_list = [pd.read_excel(arquivo) for arquivo in conjunto]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    #Z_media = np.mean(Z_pred_total, axis=1, keepdims=True)
    Z_media = Z_pred_total @ best_w
    Z_media = Z_media.reshape(-1, 1)
    df = pd.read_excel("1000_model/1000_model_4_8_0.xlsx")
    Z = df['Z'].values.reshape(-1, 1)

    mse_sup = np.mean((Z - Z_media) ** 2)
  
    r2_sup = r2_score(Z, Z_media)
    return mse_sup, r2_sup

# Função para cálculo da diversidade usando 25_model
def diversity(conjunto, best_w):
    df_list = [pd.read_excel(arquivo) for arquivo in conjunto]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]

    df = pd.read_excel("25_model/25_model_4_8_0.xlsx")
    Z = df['Z'].values.reshape(-1, 1)

    Z_pred_total = np.hstack(z_preds)
    #Z_media = np.mean(Z_pred_total, axis=1, keepdims=True)
    Z_media = Z_pred_total @ best_w
    Z_media = Z_media.reshape(-1, 1)

    error1 = sum((Z - z_pred) ** 2 for z_pred in z_preds)
    error2 = sum((z_pred - Z_media) ** 2 for z_pred in z_preds)
    error = error1 + error2

    return np.mean(error.T)

# Função para cálculo de variância, viés e covariância usando 25_model
def mse_var_bias_covar(conjunto, best_w):
    df_list = [pd.read_excel(arquivo) for arquivo in conjunto]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    #Z_media = np.mean(Z_pred_total, axis=1, keepdims=True)
    Z_media = Z_pred_total @ best_w
    Z_media = Z_media.reshape(-1, 1)
    df = pd.read_excel("25_model/25_model_4_8_0.xlsx")
    Z = df['Z'].values.reshape(-1, 1)

    var = [np.mean((z_pred - Z_media) ** 2) for z_pred in z_preds]
    var = np.mean(np.array(var))

    bias = np.mean((Z_media - Z))

    M = 10  # número de redes
    cov_sum = 0
    for i in range(M):
        for j in range(M):
            if i != j:
                f_i = z_preds[i]
                f_j = z_preds[j]
                f_i_mean = np.mean(f_i)
                f_j_mean = np.mean(f_j)
                cov_ij = np.mean((f_i - f_i_mean) * (f_j - f_j_mean))
                cov_sum += cov_ij

    covar = cov_sum / (M * (M - 1))
    r2 = r2_score(Z, Z_media)

    return ( var, bias, covar, r2)


def filtrar_combinacoes_sincronizadas(
    diretorio_25_model,
    diretorio_1000_model,
    tamanho_conjunto,
    pasta_destino,
    nome_pasta,
    limite=1000
):
    resultados = []

    # Carrega os nomes dos arquivos
    arquivos_25 = sorted([f for f in os.listdir(diretorio_25_model) if f.endswith(".xlsx")])
    arquivos_1000 = sorted([f for f in os.listdir(diretorio_1000_model) if f.endswith(".xlsx")])

    # Mapeia pelos sufixos
    nomes_25 = {
        f.replace("25_model_", ""): os.path.join(diretorio_25_model, f)
        for f in arquivos_25
    }
    nomes_1000 = {
        f.replace("1000_model_", ""): os.path.join(diretorio_1000_model, f)
        for f in arquivos_1000
    }

    # Garante apenas os nomes que existem nos dois diretórios
    nomes_comuns = sorted(set(nomes_25.keys()) & set(nomes_1000.keys()))

    if len(nomes_comuns) < tamanho_conjunto:
        raise ValueError("Poucos modelos sincronizados para o tamanho do conjunto.")

    # Pares sincronizados (tuple: 25_model_path, 1000_model_path)
    pares_sincronizados = [(nomes_25[n], nomes_1000[n]) for n in nomes_comuns]

    # Geração aleatória das combinações de índices, preservando a correspondência
    usadas = set()
    combinacoes_aleatorias = []

    while len(combinacoes_aleatorias) < limite:
        amostra = tuple(sorted(random.sample(range(len(pares_sincronizados)), tamanho_conjunto)))
        if amostra not in usadas:
            usadas.add(amostra)
            combinacoes_aleatorias.append(amostra)

    # Avalia cada combinação
    for count, combinacao_indices in enumerate(combinacoes_aleatorias, start=1):
        combinacao_25 = [pares_sincronizados[i][0] for i in combinacao_indices]
        combinacao_1000 = [pares_sincronizados[i][1] for i in combinacao_indices]

        print(f"\nCombinacao {count}:")
        print(f"25_model: {[os.path.basename(p) for p in combinacao_25]}")
        print(f"1000_model: {[os.path.basename(p) for p in combinacao_1000]}")

        Z, z_preds, N = carrega_Z_e_Zpreds(combinacao_25, "25_model/25_model_4_8_0.xlsx")
        best_w = otimiza_pesos(Z, z_preds, N)
        mse, r2_sup = calcula_mse_sup(combinacao_1000, best_w)
        erro = diversity(combinacao_25, best_w)
        var, bias, covar, r2 = mse_var_bias_covar(combinacao_25, best_w)
        MSE_var_bias = var + bias**2

        resultado = {
            "Conjunto": combinacao_25,
            "pesos": best_w,
            "MSE_sup": mse,
            "r2_sup": r2_sup,
            "Erro de Diversidade": erro,
            "r2": r2,
            "var": var,
            "bias": bias,
            "covar": covar,
            "MSE_var_bias": MSE_var_bias
        }
        resultados.append(resultado)

    salvar_resultados(resultados, pasta_destino, nome_pasta)
    return resultados



# Função para salvar todos os resultados de uma vez
def salvar_resultados(resultados, pasta_destino, nome_pasta):
    if not os.path.exists(pasta_destino):
        os.makedirs(pasta_destino)

    df = pd.DataFrame(resultados)
    caminho_arquivo = os.path.join(pasta_destino, nome_pasta)
    df.to_excel(caminho_arquivo, index=False)
    print(f"Todos os resultados salvos em: {caminho_arquivo}")

# Parâmetros
tamanho_conjunto = 10
pasta_destino = "resultados"

# Execução
resultados = filtrar_combinacoes_sincronizadas(
    diretorio_25_model,
    diretorio_1000_model,
    tamanho_conjunto=10,
    pasta_destino="resultados",
    nome_pasta= "todos_resultados_50_2.xlsx",
    limite=5000
)
 
 


Combinacao 1:
25_model: ['25_model_10_9_3.xlsx', '25_model_13_9_4.xlsx', '25_model_13_9_6.xlsx', '25_model_20_5_1.xlsx', '25_model_22_8_10.xlsx', '25_model_22_8_9.xlsx', '25_model_22_9_18.xlsx', '25_model_24_6_5.xlsx', '25_model_30_7_9.xlsx', '25_model_4_7_2.xlsx']
1000_model: ['1000_model_10_9_3.xlsx', '1000_model_13_9_4.xlsx', '1000_model_13_9_6.xlsx', '1000_model_20_5_1.xlsx', '1000_model_22_8_10.xlsx', '1000_model_22_8_9.xlsx', '1000_model_22_9_18.xlsx', '1000_model_24_6_5.xlsx', '1000_model_30_7_9.xlsx', '1000_model_4_7_2.xlsx']

Combinacao 2:
25_model: ['25_model_10_9_0.xlsx', '25_model_10_9_1.xlsx', '25_model_12_6_0.xlsx', '25_model_20_5_1.xlsx', '25_model_22_9_19.xlsx', '25_model_22_9_22.xlsx', '25_model_30_7_6.xlsx', '25_model_30_7_8.xlsx', '25_model_30_7_9.xlsx', '25_model_9_9_4.xlsx']
1000_model: ['1000_model_10_9_0.xlsx', '1000_model_10_9_1.xlsx', '1000_model_12_6_0.xlsx', '1000_model_20_5_1.xlsx', '1000_model_22_9_19.xlsx', '1000_model_22_9_22.xlsx', '1000_model_30_7_6.xl

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
import random 

# =====================================================
# HIPERPARÂMETROS A SEREM VARIADOS
# =====================================================
lambda_regs = [1e-6, 5e-6, 1e-5, 1e-3]
lrs = [1e-5, 1e-2, 1e-3]
n_epocas_list = [500000, 50000, 1000]

patience = 500
min_delta = 1e-2

# =====================================================
# DIRETÓRIOS DOS MODELOS
# =====================================================
diretorio_25_model = "/workspaces/IC-RNA-2025/Peaks-dataset-article/redes-ensemble-s/metricas_ensemble_50_redes_1/25_model"
diretorio_1000_model = "/workspaces/IC-RNA-2025/Peaks-dataset-article/redes-ensemble-s/metricas_ensemble_50_redes_1/1000_model"


# =====================================================
# 1) CARREGAMENTO EFICIENTE (CACHE EM MEMÓRIA)
# =====================================================
def carregar_modelos_em_memoria(diretorio):
    modelos = {}
    for f in os.listdir(diretorio):
        if f.endswith(".xlsx"):
            df = pd.read_excel(os.path.join(diretorio, f), usecols=["Z", "Z_pred"])
            modelos[f] = df
    return modelos


modelos_25 = carregar_modelos_em_memoria(diretorio_25_model)
modelos_1000 = carregar_modelos_em_memoria(diretorio_1000_model)


# =====================================================
# 2) FUNÇÕES NUMÉRICAS
# =====================================================
def carrega_Z_e_Zpreds(conjunto_25, modelos_25, arquivo_Z_base):
    df_list = [modelos_25[os.path.basename(f)] for f in conjunto_25]
    z_preds = np.hstack([df['Z_pred'].values.reshape(-1,1) for df in df_list])
    Z = modelos_25[os.path.basename(arquivo_Z_base)]['Z'].values.reshape(-1,1)
    return Z, z_preds, Z.shape[0]


def otimiza_pesos(Z, z_preds, N, lambda_reg, lr, n_epocas):
    a = np.random.uniform(0, 1, z_preds.shape[1])
    best_mse_25 = -np.inf
    best_w = None
    patience_counter = 0

    for _ in range(n_epocas):
        w = np.exp(a) / np.sum(np.exp(a))
        yhat = z_preds @ w

        residuo = Z.ravel() - yhat
        mse = np.mean(residuo**2)

        gmse = (-2.0 / N) * (z_preds.T @ residuo)
        s = gmse @ w
        grad_a = w * (gmse - s) + 2.0 * lambda_reg * a

        a -= lr * grad_a

        # critério: MAIOR mse_25
        if mse > best_mse_25 + min_delta:
            best_mse_25 = mse
            best_w = w.copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    return best_w


def calcula_mse_sup(conjunto_1000, best_w, modelos_1000):
    df_list = [modelos_1000[os.path.basename(f)] for f in conjunto_1000]
    Z_pred_total = np.hstack([df['Z_pred'].values.reshape(-1, 1) for df in df_list])
    Z_media = (Z_pred_total @ best_w).reshape(-1, 1)

    Z = df_list[0]['Z'].values.reshape(-1, 1)

    mse_sup = np.mean((Z - Z_media) ** 2)
    r2_sup = r2_score(Z, Z_media)
    return mse_sup, r2_sup


def diversity(conjunto_25, best_w, modelos_25):
    df_list = [modelos_25[os.path.basename(f)] for f in conjunto_25]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    Z_media = (Z_pred_total @ best_w).reshape(-1, 1)

    Z = df_list[0]['Z'].values.reshape(-1, 1)

    error1 = sum((Z - z_pred) ** 2 for z_pred in z_preds)
    error2 = sum((z_pred - Z_media) ** 2 for z_pred in z_preds)
    error = error1 + error2

    return np.mean(error.T)


def mse_var_bias_covar(conjunto_25, best_w, modelos_25):
    df_list = [modelos_25[os.path.basename(f)] for f in conjunto_25]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    Z_media = (Z_pred_total @ best_w).reshape(-1, 1)

    Z = df_list[0]['Z'].values.reshape(-1, 1)

    var = np.mean([(np.mean((z_pred - Z_media) ** 2)) for z_pred in z_preds])
    bias = np.mean(Z_media - Z)

    M = len(z_preds)
    cov_sum = 0.0
    for i in range(M):
        fi = z_preds[i] - np.mean(z_preds[i])
        for j in range(i + 1, M):
            fj = z_preds[j] - np.mean(z_preds[j])
            cov_sum += np.mean(fi * fj)

    covar = 2 * cov_sum / (M * (M - 1))
    r2 = r2_score(Z, Z_media)

    return var, bias, covar, r2


# =====================================================
# 3) FUNÇÃO PRINCIPAL COM GRID SEARCH
# =====================================================
def filtrar_combinacoes_sincronizadas(
    diretorio_25_model,
    diretorio_1000_model,
    tamanho_conjunto,
    pasta_destino,
    nome_pasta,
    limite=1000
):
    resultados = []

    arquivos_25 = sorted([f for f in modelos_25.keys()])
    arquivos_1000 = sorted([f for f in modelos_1000.keys()])

    nomes_25 = {f.replace("25_model_", ""): f for f in arquivos_25}
    nomes_1000 = {f.replace("1000_model_", ""): f for f in arquivos_1000}

    nomes_comuns = sorted(set(nomes_25.keys()) & set(nomes_1000.keys()))

    if len(nomes_comuns) < tamanho_conjunto:
        raise ValueError("Poucos modelos sincronizados para o tamanho do conjunto.")

    pares_sincronizados = [(nomes_25[n], nomes_1000[n]) for n in nomes_comuns]

    usadas = set()
    combinacoes_aleatorias = []

    while len(combinacoes_aleatorias) < limite:
        amostra = tuple(sorted(random.sample(range(len(pares_sincronizados)), tamanho_conjunto)))
        if amostra not in usadas:
            usadas.add(amostra)
            combinacoes_aleatorias.append(amostra)

    for count, combinacao_indices in enumerate(combinacoes_aleatorias, start=1):
        combinacao_25 = [pares_sincronizados[i][0] for i in combinacao_indices]
        combinacao_1000 = [pares_sincronizados[i][1] for i in combinacao_indices]

        print(f"\nCombinacao {count}:")
        print(f"25_model: {combinacao_25}")
        print(f"1000_model: {combinacao_1000}")

        Z, z_preds, N = carrega_Z_e_Zpreds(
            combinacao_25,
            modelos_25,
            combinacao_25[0]
        )

        # 🔁 VARREDURA DE HIPERPARÂMETROS
        for lambda_reg in lambda_regs:
            for lr in lrs:
                for n_epocas in n_epocas_list:

                    #print(f"  Testando λ={lambda_reg}, lr={lr}, épocas={n_epocas}")

                    best_w = otimiza_pesos(Z, z_preds, N, lambda_reg, lr, n_epocas)

                    mse, r2_sup = calcula_mse_sup(combinacao_1000, best_w, modelos_1000)
                    erro = diversity(combinacao_25, best_w, modelos_25)
                    var, bias, covar, r2 = mse_var_bias_covar(combinacao_25, best_w, modelos_25)
                    MSE_var_bias = var + bias**2

                    resultado = {
                        "Conjunto": combinacao_25,
                        "lambda_reg": lambda_reg,
                        "lr": lr,
                        "n_epocas": n_epocas,
                        "pesos": best_w,
                        "MSE_sup": mse,
                        "r2_sup": r2_sup,
                        "Erro de Diversidade": erro,
                        "r2": r2,
                        "var": var,
                        "bias": bias,
                        "covar": covar,
                        "MSE_var_bias": MSE_var_bias
                    }
                    resultados.append(resultado)

    salvar_resultados(resultados, pasta_destino, nome_pasta)
    return resultados


# =====================================================
# 4) SALVAMENTO
# =====================================================
def salvar_resultados(resultados, pasta_destino, nome_pasta):
    if not os.path.exists(pasta_destino):
        os.makedirs(pasta_destino)

    df = pd.DataFrame(resultados)
    caminho_arquivo = os.path.join(pasta_destino, nome_pasta)
    df.to_excel(caminho_arquivo, index=False)
    print(f"Todos os resultados salvos em: {caminho_arquivo}")


# =====================================================
# 5) EXECUÇÃO
# =====================================================
resultados = filtrar_combinacoes_sincronizadas(
    diretorio_25_model,
    diretorio_1000_model,
    tamanho_conjunto=10,
    pasta_destino="resultados",
    nome_pasta="todos_resultados_50_lm.xlsx",
    limite=5000
)



Combinacao 1:
25_model: ['25_model_10_9_3.xlsx', '25_model_12_6_0.xlsx', '25_model_13_9_5.xlsx', '25_model_22_8_8.xlsx', '25_model_22_9_18.xlsx', '25_model_22_9_22.xlsx', '25_model_24_9_0.xlsx', '25_model_30_7_9.xlsx', '25_model_34_6_0.xlsx', '25_model_4_7_2.xlsx']
1000_model: ['1000_model_10_9_3.xlsx', '1000_model_12_6_0.xlsx', '1000_model_13_9_5.xlsx', '1000_model_22_8_8.xlsx', '1000_model_22_9_18.xlsx', '1000_model_22_9_22.xlsx', '1000_model_24_9_0.xlsx', '1000_model_30_7_9.xlsx', '1000_model_34_6_0.xlsx', '1000_model_4_7_2.xlsx']

Combinacao 2:
25_model: ['25_model_10_9_0.xlsx', '25_model_10_9_1.xlsx', '25_model_13_9_3.xlsx', '25_model_20_5_1.xlsx', '25_model_20_9_7.xlsx', '25_model_22_8_10.xlsx', '25_model_22_9_18.xlsx', '25_model_22_9_20.xlsx', '25_model_25_7_6.xlsx', '25_model_30_7_5.xlsx']
1000_model: ['1000_model_10_9_0.xlsx', '1000_model_10_9_1.xlsx', '1000_model_13_9_3.xlsx', '1000_model_20_5_1.xlsx', '1000_model_20_9_7.xlsx', '1000_model_22_8_10.xlsx', '1000_model_22_9_18.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
import random 

# =====================================================
# HIPERPARÂMETROS A SEREM VARIADOS
# =====================================================
lambda_regs = [1e-6, 5e-6, 1e-5, 1e-3]
lrs = [1e-5, 1e-2, 1e-3]
n_epocas_list = [500000, 50000, 1000]

patience = 500
min_delta = 1e-2

# =====================================================
# DIRETÓRIOS DOS MODELOS
# =====================================================
diretorio_25_model = "/workspaces/IC-RNA-2025/Peaks-dataset-article/redes-ensemble-s/metricas_ensemble_50_redes_1/25_model"
diretorio_1000_model = "/workspaces/IC-RNA-2025/Peaks-dataset-article/redes-ensemble-s/metricas_ensemble_50_redes_1/1000_model"


# =====================================================
# 1) CARREGAMENTO EFICIENTE (CACHE EM MEMÓRIA)
# =====================================================
def carregar_modelos_em_memoria(diretorio):
    modelos = {}
    for f in os.listdir(diretorio):
        if f.endswith(".xlsx"):
            df = pd.read_excel(os.path.join(diretorio, f), usecols=["Z", "Z_pred"])
            modelos[f] = df
    return modelos


modelos_25 = carregar_modelos_em_memoria(diretorio_25_model)
modelos_1000 = carregar_modelos_em_memoria(diretorio_1000_model)


# =====================================================
# 2) FUNÇÕES NUMÉRICAS
# =====================================================
def carrega_Z_e_Zpreds(conjunto_25, modelos_25, arquivo_Z_base):
    df_list = [modelos_25[os.path.basename(f)] for f in conjunto_25]
    z_preds = np.hstack([df['Z_pred'].values.reshape(-1,1) for df in df_list])
    Z = modelos_25[os.path.basename(arquivo_Z_base)]['Z'].values.reshape(-1,1)
    return Z, z_preds, Z.shape[0]


def otimiza_pesos(Z, z_preds, N, lambda_reg, lr, n_epocas):
    a = np.random.uniform(0, 1, z_preds.shape[1])
    best_mse_25 = -np.inf
    best_w = None
    patience_counter = 0

    for _ in range(n_epocas):
        w = np.exp(a) / np.sum(np.exp(a))
        yhat = z_preds @ w

        residuo = Z.ravel() - yhat
        mse = np.mean(residuo**2)

        gmse = (-2.0 / N) * (z_preds.T @ residuo)
        s = gmse @ w
        grad_a = w * (gmse - s) + 2.0 * lambda_reg * a

        a -= lr * grad_a

        # critério: MAIOR mse_25
        if mse > best_mse_25 + min_delta:
            best_mse_25 = mse
            best_w = w.copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    return best_w


def calcula_mse_sup(conjunto_1000, best_w, modelos_1000):
    df_list = [modelos_1000[os.path.basename(f)] for f in conjunto_1000]
    Z_pred_total = np.hstack([df['Z_pred'].values.reshape(-1, 1) for df in df_list])
    Z_media = (Z_pred_total @ best_w).reshape(-1, 1)

    Z = df_list[0]['Z'].values.reshape(-1, 1)

    mse_sup = np.mean((Z - Z_media) ** 2)
    r2_sup = r2_score(Z, Z_media)
    return mse_sup, r2_sup


def diversity(conjunto_25, best_w, modelos_25):
    df_list = [modelos_25[os.path.basename(f)] for f in conjunto_25]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    Z_media = (Z_pred_total @ best_w).reshape(-1, 1)

    Z = df_list[0]['Z'].values.reshape(-1, 1)

    error1 = sum((Z - z_pred) ** 2 for z_pred in z_preds)
    error2 = sum((z_pred - Z_media) ** 2 for z_pred in z_preds)
    error = error1 + error2

    return np.mean(error.T)


def mse_var_bias_covar(conjunto_25, best_w, modelos_25):
    df_list = [modelos_25[os.path.basename(f)] for f in conjunto_25]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    Z_media = (Z_pred_total @ best_w).reshape(-1, 1)

    Z = df_list[0]['Z'].values.reshape(-1, 1)

    var = np.mean([(np.mean((z_pred - Z_media) ** 2)) for z_pred in z_preds])
    bias = np.mean(Z_media - Z)

    M = len(z_preds)
    cov_sum = 0.0
    for i in range(M):
        fi = z_preds[i] - np.mean(z_preds[i])
        for j in range(i + 1, M):
            fj = z_preds[j] - np.mean(z_preds[j])
            cov_sum += np.mean(fi * fj)

    covar = 2 * cov_sum / (M * (M - 1))
    r2 = r2_score(Z, Z_media)

    return var, bias, covar, r2


# =====================================================
# 3) FUNÇÃO PRINCIPAL COM GRID SEARCH
# =====================================================
def filtrar_combinacoes_sincronizadas(
    diretorio_25_model,
    diretorio_1000_model,
    tamanho_conjunto,
    pasta_destino,
    nome_pasta,
    limite=1000
):
    resultados = []

    arquivos_25 = sorted([f for f in modelos_25.keys()])
    arquivos_1000 = sorted([f for f in modelos_1000.keys()])

    nomes_25 = {f.replace("25_model_", ""): f for f in arquivos_25}
    nomes_1000 = {f.replace("1000_model_", ""): f for f in arquivos_1000}

    nomes_comuns = sorted(set(nomes_25.keys()) & set(nomes_1000.keys()))

    if len(nomes_comuns) < tamanho_conjunto:
        raise ValueError("Poucos modelos sincronizados para o tamanho do conjunto.")

    pares_sincronizados = [(nomes_25[n], nomes_1000[n]) for n in nomes_comuns]

    usadas = set()
    combinacoes_aleatorias = []

    while len(combinacoes_aleatorias) < limite:
        amostra = tuple(sorted(random.sample(range(len(pares_sincronizados)), tamanho_conjunto)))
        if amostra not in usadas:
            usadas.add(amostra)
            combinacoes_aleatorias.append(amostra)

    for count, combinacao_indices in enumerate(combinacoes_aleatorias, start=1):
        combinacao_25 = [pares_sincronizados[i][0] for i in combinacao_indices]
        combinacao_1000 = [pares_sincronizados[i][1] for i in combinacao_indices]

        print(f"\nCombinacao {count}:")
        print(f"25_model: {combinacao_25}")
        print(f"1000_model: {combinacao_1000}")

        Z, z_preds, N = carrega_Z_e_Zpreds(
            combinacao_25,
            modelos_25,
            combinacao_25[0]
        )

        # 🔁 VARREDURA DE HIPERPARÂMETROS
        for lambda_reg in lambda_regs:
            for lr in lrs:
                for n_epocas in n_epocas_list:

                    #print(f"  Testando λ={lambda_reg}, lr={lr}, épocas={n_epocas}")

                    best_w = otimiza_pesos(Z, z_preds, N, lambda_reg, lr, n_epocas)

                    mse, r2_sup = calcula_mse_sup(combinacao_1000, best_w, modelos_1000)
                    erro = diversity(combinacao_25, best_w, modelos_25)
                    var, bias, covar, r2 = mse_var_bias_covar(combinacao_25, best_w, modelos_25)
                    MSE_var_bias = var + bias**2

                    resultado = {
                        "Conjunto": combinacao_25,
                        "lambda_reg": lambda_reg,
                        "lr": lr,
                        "n_epocas": n_epocas,
                        "pesos": best_w,
                        "MSE_sup": mse,
                        "r2_sup": r2_sup,
                        "Erro de Diversidade": erro,
                        "r2": r2,
                        "var": var,
                        "bias": bias,
                        "covar": covar,
                        "MSE_var_bias": MSE_var_bias
                    }
                    resultados.append(resultado)

    salvar_resultados(resultados, pasta_destino, nome_pasta)
    return resultados


# =====================================================
# 4) SALVAMENTO
# =====================================================
def salvar_resultados(resultados, pasta_destino, nome_pasta):
    if not os.path.exists(pasta_destino):
        os.makedirs(pasta_destino)

    df = pd.DataFrame(resultados)
    caminho_arquivo = os.path.join(pasta_destino, nome_pasta)
    df.to_excel(caminho_arquivo, index=False)
    print(f"Todos os resultados salvos em: {caminho_arquivo}")


# =====================================================
# 5) EXECUÇÃO
# =====================================================
resultados = filtrar_combinacoes_sincronizadas(
    diretorio_25_model,
    diretorio_1000_model,
    tamanho_conjunto=10,
    pasta_destino="resultados",
    nome_pasta="todos_resultados_50_lm.xlsx",
    limite=5000
)


In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
import random 

# =====================================================
# HIPERPARÂMETROS A SEREM VARIADOS
# =====================================================
lambda_regs = [1e-6, 5e-6, 1e-5, 1e-3]
lrs = [1e-5, 1e-2, 1e-3]
n_epocas_list = [500000, 50000, 1000]

patience = 500
min_delta = 1e-2

# =====================================================
# DIRETÓRIOS DOS MODELOS
# =====================================================
diretorio_25_model = "/workspaces/IC-RNA-2025/Peaks-dataset-article/redes-ensemble-s/metricas_ensemble_50_redes_1/25_model"
diretorio_1000_model = "/workspaces/IC-RNA-2025/Peaks-dataset-article/redes-ensemble-s/metricas_ensemble_50_redes_1/1000_model"


# =====================================================
# 1) CARREGAMENTO EFICIENTE (CACHE EM MEMÓRIA)
# =====================================================
def carregar_modelos_em_memoria(diretorio):
    modelos = {}
    for f in os.listdir(diretorio):
        if f.endswith(".xlsx"):
            df = pd.read_excel(os.path.join(diretorio, f), usecols=["Z", "Z_pred"])
            modelos[f] = df
    return modelos


modelos_25 = carregar_modelos_em_memoria(diretorio_25_model)
modelos_1000 = carregar_modelos_em_memoria(diretorio_1000_model)


# =====================================================
# 2) FUNÇÕES NUMÉRICAS
# =====================================================
def carrega_Z_e_Zpreds(conjunto_25, modelos_25, arquivo_Z_base):
    df_list = [modelos_25[os.path.basename(f)] for f in conjunto_25]
    z_preds = np.hstack([df['Z_pred'].values.reshape(-1,1) for df in df_list])
    Z = modelos_25[os.path.basename(arquivo_Z_base)]['Z'].values.reshape(-1,1)
    return Z, z_preds, Z.shape[0]

def otimiza_pesos(Z, z_preds, N, lambda_reg, lr, n_epocas):
    M = z_preds.shape[1]
    a = np.random.uniform(0, 1, M)

    best_mse_25 = -np.inf
    best_w = None
    patience_counter = 0
    mu = 1e-3  # damping inicial

    for _ in range(n_epocas):
        # Softmax para impor pesos >=0 e soma = 1
        w = np.exp(a) / np.sum(np.exp(a))
        yhat = z_preds @ w

        residuo = Z.ravel() - yhat
        mse = np.mean(residuo ** 2)

        # Gradiente em relação a w
        gmse = (-2.0 / N) * (z_preds.T @ residuo)

        # Jacobiano do softmax
        s = gmse @ w
        grad_a = w * (gmse - s) + 2.0 * lambda_reg * a

        # Hessiana aproximada (Gauss–Newton)
        H = np.outer(w, w) * np.mean(z_preds.T @ z_preds) + 2.0 * lambda_reg * np.eye(M)

        # Atualização LM
        try:
            delta = np.linalg.solve(H + mu * np.eye(M), grad_a)
        except np.linalg.LinAlgError:
            break

        a_new = a + delta

        # Avalia novo ponto
        w_new = np.exp(a_new) / np.sum(np.exp(a_new))
        yhat_new = z_preds @ w_new
        residuo_new = Z.ravel() - yhat_new
        mse_new = np.mean(residuo_new ** 2)

        if mse_new > mse:  # melhora → aceita passo
            a = a_new
            mu *= 0.7
        else:  # piora → aumenta damping
            mu *= 2.0

        # critério: MAIOR mse_25 (como no seu código)
        if mse_new > best_mse_25 + min_delta:
            best_mse_25 = mse_new
            best_w = w_new.copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    return best_w

def calcula_mse_sup(conjunto_1000, best_w, modelos_1000):
    df_list = [modelos_1000[os.path.basename(f)] for f in conjunto_1000]
    Z_pred_total = np.hstack([df['Z_pred'].values.reshape(-1, 1) for df in df_list])
    Z_media = (Z_pred_total @ best_w).reshape(-1, 1)

    Z = df_list[0]['Z'].values.reshape(-1, 1)

    mse_sup = np.mean((Z - Z_media) ** 2)
    r2_sup = r2_score(Z, Z_media)
    return mse_sup, r2_sup


def diversity(conjunto_25, best_w, modelos_25):
    df_list = [modelos_25[os.path.basename(f)] for f in conjunto_25]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    Z_media = (Z_pred_total @ best_w).reshape(-1, 1)

    Z = df_list[0]['Z'].values.reshape(-1, 1)

    error1 = sum((Z - z_pred) ** 2 for z_pred in z_preds)
    error2 = sum((z_pred - Z_media) ** 2 for z_pred in z_preds)
    error = error1 + error2

    return np.mean(error.T)


def mse_var_bias_covar(conjunto_25, best_w, modelos_25):
    df_list = [modelos_25[os.path.basename(f)] for f in conjunto_25]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    Z_media = (Z_pred_total @ best_w).reshape(-1, 1)

    Z = df_list[0]['Z'].values.reshape(-1, 1)

    var = np.mean([(np.mean((z_pred - Z_media) ** 2)) for z_pred in z_preds])
    bias = np.mean(Z_media - Z)

    M = len(z_preds)
    cov_sum = 0.0
    for i in range(M):
        fi = z_preds[i] - np.mean(z_preds[i])
        for j in range(i + 1, M):
            fj = z_preds[j] - np.mean(z_preds[j])
            cov_sum += np.mean(fi * fj)

    covar = 2 * cov_sum / (M * (M - 1))
    r2 = r2_score(Z, Z_media)

    return var, bias, covar, r2


# =====================================================
# 3) FUNÇÃO PRINCIPAL COM GRID SEARCH
# =====================================================
def filtrar_combinacoes_sincronizadas(
    diretorio_25_model,
    diretorio_1000_model,
    tamanho_conjunto,
    pasta_destino,
    nome_pasta,
    limite=1000
):
    resultados = []

    arquivos_25 = sorted([f for f in modelos_25.keys()])
    arquivos_1000 = sorted([f for f in modelos_1000.keys()])

    nomes_25 = {f.replace("25_model_", ""): f for f in arquivos_25}
    nomes_1000 = {f.replace("1000_model_", ""): f for f in arquivos_1000}

    nomes_comuns = sorted(set(nomes_25.keys()) & set(nomes_1000.keys()))

    if len(nomes_comuns) < tamanho_conjunto:
        raise ValueError("Poucos modelos sincronizados para o tamanho do conjunto.")

    pares_sincronizados = [(nomes_25[n], nomes_1000[n]) for n in nomes_comuns]

    usadas = set()
    combinacoes_aleatorias = []

    while len(combinacoes_aleatorias) < limite:
        amostra = tuple(sorted(random.sample(range(len(pares_sincronizados)), tamanho_conjunto)))
        if amostra not in usadas:
            usadas.add(amostra)
            combinacoes_aleatorias.append(amostra)

    for count, combinacao_indices in enumerate(combinacoes_aleatorias, start=1):
        combinacao_25 = [pares_sincronizados[i][0] for i in combinacao_indices]
        combinacao_1000 = [pares_sincronizados[i][1] for i in combinacao_indices]

        print(f"\nCombinacao {count}:")
        print(f"25_model: {combinacao_25}")
        print(f"1000_model: {combinacao_1000}")

        Z, z_preds, N = carrega_Z_e_Zpreds(
            combinacao_25,
            modelos_25,
            combinacao_25[0]
        )

        # 🔁 VARREDURA DE HIPERPARÂMETROS
        for lambda_reg in lambda_regs:
            for lr in lrs:
                for n_epocas in n_epocas_list:

                    #print(f"  Testando λ={lambda_reg}, lr={lr}, épocas={n_epocas}")

                    best_w = otimiza_pesos(Z, z_preds, N, lambda_reg, lr, n_epocas)

                    mse, r2_sup = calcula_mse_sup(combinacao_1000, best_w, modelos_1000)
                    erro = diversity(combinacao_25, best_w, modelos_25)
                    var, bias, covar, r2 = mse_var_bias_covar(combinacao_25, best_w, modelos_25)
                    MSE_var_bias = var + bias**2

                    resultado = {
                        "Conjunto": combinacao_25,
                        "lambda_reg": lambda_reg,
                        "lr": lr,
                        "n_epocas": n_epocas,
                        "pesos": best_w,
                        "MSE_sup": mse,
                        "r2_sup": r2_sup,
                        "Erro de Diversidade": erro,
                        "r2": r2,
                        "var": var,
                        "bias": bias,
                        "covar": covar,
                        "MSE_var_bias": MSE_var_bias
                    }
                    resultados.append(resultado)

    salvar_resultados(resultados, pasta_destino, nome_pasta)
    return resultados


# =====================================================
# 4) SALVAMENTO
# =====================================================
def salvar_resultados(resultados, pasta_destino, nome_pasta):
    if not os.path.exists(pasta_destino):
        os.makedirs(pasta_destino)

    df = pd.DataFrame(resultados)
    caminho_arquivo = os.path.join(pasta_destino, nome_pasta)
    df.to_excel(caminho_arquivo, index=False)
    print(f"Todos os resultados salvos em: {caminho_arquivo}")


# =====================================================
# 5) EXECUÇÃO
# =====================================================
resultados = filtrar_combinacoes_sincronizadas(
    diretorio_25_model,
    diretorio_1000_model,
    tamanho_conjunto=10,
    pasta_destino="resultados",
    nome_pasta="todos_resultados_50_lm.xlsx",
    limite=5000
)




Combinacao 1:
25_model: ['25_model_12_6_0.xlsx', '25_model_13_9_2.xlsx', '25_model_13_9_3.xlsx', '25_model_13_9_5.xlsx', '25_model_20_5_1.xlsx', '25_model_20_9_6.xlsx', '25_model_20_9_7.xlsx', '25_model_22_9_21.xlsx', '25_model_34_6_1.xlsx', '25_model_9_9_2.xlsx']
1000_model: ['1000_model_12_6_0.xlsx', '1000_model_13_9_2.xlsx', '1000_model_13_9_3.xlsx', '1000_model_13_9_5.xlsx', '1000_model_20_5_1.xlsx', '1000_model_20_9_6.xlsx', '1000_model_20_9_7.xlsx', '1000_model_22_9_21.xlsx', '1000_model_34_6_1.xlsx', '1000_model_9_9_2.xlsx']

Combinacao 2:
25_model: ['25_model_10_9_3.xlsx', '25_model_13_9_6.xlsx', '25_model_20_8_0.xlsx', '25_model_20_8_1.xlsx', '25_model_22_8_11.xlsx', '25_model_22_8_9.xlsx', '25_model_22_9_14.xlsx', '25_model_22_9_22.xlsx', '25_model_34_6_0.xlsx', '25_model_9_9_5.xlsx']
1000_model: ['1000_model_10_9_3.xlsx', '1000_model_13_9_6.xlsx', '1000_model_20_8_0.xlsx', '1000_model_20_8_1.xlsx', '1000_model_22_8_11.xlsx', '1000_model_22_8_9.xlsx', '1000_model_22_9_14.xls

KeyboardInterrupt: 